In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score


import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [4]:
encoder = OneHotEncoder(sparse_output=False)

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [7]:
alphas_alt = [14.5, 14.6, 14.7, 14.8, 14.9, 15, 15.1, 15.2, 15.3, 15.4, 15.5]
alphas2 = [5e-05, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008]
e_alphas = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007]
e_l1ratio = [0.8, 0.85, 0.9, 0.95, 0.99, 1]

In [8]:
ridge = make_pipeline(RobustScaler(), 
                      RidgeCV(alphas=alphas_alt))

lasso = make_pipeline(RobustScaler(), 
                      LassoCV(max_iter=10000, 
                              alphas=alphas2, 
                              random_state=42))

elasticnet = make_pipeline(RobustScaler(), 
                           ElasticNetCV(max_iter=10000, 
                                        alphas=e_alphas, 
                                        l1_ratio=e_l1ratio))

randomforest=make_pipeline(RobustScaler(), 
                           RandomForestRegressor(max_depth=9, 
                                                 max_features=None, 
                                                 max_leaf_nodes=9, 
                                                 n_estimators=100))
svr = make_pipeline(RobustScaler(), 
                    SVR(kernel='rbf', 
                        C=1, 
                        epsilon=0.1, 
                        gamma=0.1))

In [9]:
gbr = GradientBoostingRegressor(n_estimators=3000, 
                                learning_rate=0.05, 
                                max_depth=4, 
                                max_features='sqrt', 
                                min_samples_leaf=15, 
                                min_samples_split=10, 
                                loss='huber', 
                                random_state =42)    

lightgbm = LGBMRegressor(colsample_bytree=0.9119276182779381, 
                         drop_rate=0.2870902369968208,
                         learning_rate=0.09797055466390917, 
                         max_bin=29, 
                         max_depth=3,
                         max_drop=111, 
                         min_child_samples=33, 
                         min_data_in_leaf=68,
                         n_estimators=277, 
                         num_leaves=25, 
                         reg_alpha=0.12688048100404745,
                         reg_lambda=0.889850076976295, 
                         skip_drop=0.5462032763564192,
                         verbosity=-1)

xgboost = XGBRegressor(device="cuda",
                       max_depth=3,  
                       colsample_bytree=0.5, 
                       subsample=0.8, 
                       n_estimators=10_000,  
                       learning_rate=0.01,
                       objective='reg:logistic',
                        #eval_metric='cox-nloglik',
                        #objective='survival:cox',
                       enable_categorical=True,
                       min_child_weight=5
                    )

catboost = CatBoostRegressor(learning_rate=0.009,
                             depth=5,
                             l2_leaf_reg=3.5,
                             min_child_samples=32,
                             grow_policy='Depthwise',
                             iterations=8000,
                             eval_metric='RMSE',
                             od_type='Iter',
                             od_wait=50,
                             random_state=42,
                             logging_level='Silent')

In [10]:
model = StackingRegressor(estimators=[('catboost', catboost),
                                      ('ridge', ridge), ('lasso', lasso), 
                                      ('elasticnet', elasticnet), 
                                      ('svr', svr), ('gbr', gbr),
                                      ('xgboost', xgboost), ('lightgbm', lightgbm)],
                                      final_estimator=catboost)

In [11]:
model.fit(train_x, train_y)

/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:09:21] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:34:07] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:34:38] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:35:10] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [20:35:41] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.


StackingRegressor(estimators=[('catboost',
                               <catboost.core.CatBoostRegressor object at 0x7a3f7ade1a80>),
                              ('ridge',
                               Pipeline(steps=[('robustscaler', RobustScaler()),
                                               ('ridgecv',
                                                RidgeCV(alphas=[14.5, 14.6,
                                                                14.7, 14.8,
                                                                14.9, 15, 15.1,
                                                                15.2, 15.3,
                                                                15.4,
                                                                15.5]))])),
                              ('lasso',
                               Pipeline(steps=[('robustscaler', RobustScaler()),
                                               ('lassocv',
                                                LassoCV(alphas=[5e-05, 0.0001,
                                                                0.0002, 0.00...
                               LGBMRegressor(colsample_bytree=0.9119276182779381,
                                             drop_rate=0.2870902369968208,
                                             learning_rate=0.09797055466390917,
                                             max_bin=29, max_depth=3,
                                             max_drop=111, min_child_samples=33,
                                             min_data_in_leaf=68,
                                             n_estimators=277, num_leaves=25,
                                             reg_alpha=0.12688048100404745,
                                             reg_lambda=0.889850076976295,
                                             skip_drop=0.5462032763564192,
                                             verbosity=-1))],
                  final_estimator=<catboost.core.CatBoostRegressor object at 0x7a3f7ade1a80>)

In [12]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']

In [13]:

test_predictions = model.predict(test)

In [14]:
test_predictions

array([0.14153056, 0.90780074, 0.51076124])

In [15]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.141531
1,28801,0.907801
2,28802,0.510761


In [16]:
submission.to_csv('submission.csv', index = False)